# Tech Challenge Fase 3 - Previsão de Casos de Dengue

## 📋 Notebook 1: Definição do Problema e Coleta de Dados

### 📌 Objetivo
Este notebook aborda as primeiras etapas do Tech Challenge:
1. **Definição do problema**: Regressão para prever casos de dengue
2. **Coleta de dados**: Análise inicial do dataset disponível
3. **Armazenamento**: Dataset estruturado em formato CSV

### 🎯 Problema de Negócio
Prever o número de casos de dengue por município utilizando dados climáticos e de saneamento para auxiliar na gestão de saúde pública e alocação de recursos.

**Tipo do Problema:** Regressão (prever valores numéricos contínuos)

In [1]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações para visualização
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


## 📊 Carregamento e Análise Inicial dos Dados

In [ ]:
# Carregamento do dataset
df = pd.read_csv('dados_dengue_clima_saneamento_2014_2025.csv')

# Extrair ano e mês da coluna periodo
df['data'] = pd.to_datetime(df['periodo'])
df['Ano'] = df['data'].dt.year
df['Mês'] = df['data'].dt.month

print(f"📋 Informações básicas do dataset:")
print(f"   • Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"   • Período: {df['Ano'].min()} - {df['Ano'].max()}")
print(f"   • Estados (COD_UF): {df['COD_UF'].nunique()} únicos")
print(f"   • Total de casos no período: {df['Quantidade de Casos'].sum():,}")

# Visualização das primeiras linhas
print("\n📄 Primeiras 5 linhas do dataset:")
df.head()

📋 Informações básicas do dataset:
   • Dimensões: 3584 linhas x 20 colunas


KeyError: 'Ano'

In [ ]:
# Análise dos tipos de dados e valores ausentes
print("🔍 Informações sobre o dataset:")
print(df.info())

print("\n📊 Estatísticas descritivas da variável alvo:")
print(df['Quantidade de Casos'].describe())

print("\n❌ Valores ausentes por coluna:")
missing_values = df.isnull().sum()
if missing_values.sum() > 0:
    print(missing_values[missing_values > 0])
else:
    print("✅ Não há valores ausentes no dataset!")

## 📈 Análise Exploratória dos Dados

In [ ]:
# Análise temporal dos casos de dengue
plt.figure(figsize=(15, 5))

# Agregação por ano
casos_por_ano = df.groupby('Ano')['Quantidade de Casos'].sum()
plt.subplot(1, 2, 1)
casos_por_ano.plot(kind='bar', color='skyblue')
plt.title('Total de Casos de Dengue por Ano')
plt.xlabel('Ano')
plt.ylabel('Casos')
plt.xticks(rotation=45)

# Agregação por mês
casos_por_mes = df.groupby('Mês')['Quantidade de Casos'].sum()
plt.subplot(1, 2, 2)
casos_por_mes.plot(kind='bar', color='lightcoral')
plt.title('Total de Casos de Dengue por Mês')
plt.xlabel('Mês')
plt.ylabel('Casos')

plt.tight_layout()
plt.show()

print(f"📊 Anos com maior número de casos:")
print(casos_por_ano.sort_values(ascending=False).head(3))

In [ ]:
# Análise por estados
plt.figure(figsize=(15, 8))

# Top 10 estados com mais casos
casos_por_estado = df.groupby('COD_UF')['Quantidade de Casos'].sum().sort_values(ascending=False)
plt.subplot(2, 1, 1)
casos_por_estado.head(10).plot(kind='bar', color='mediumseagreen')
plt.title('Top 10 Estados com Mais Casos de Dengue (Total)')
plt.xlabel('Estado (COD_UF)')
plt.ylabel('Total de Casos')

# Distribuição dos casos
plt.subplot(2, 1, 2)
plt.hist(df['Quantidade de Casos'], bins=50, color='orange', alpha=0.7, edgecolor='black')
plt.title('Distribuição dos Casos de Dengue')
plt.xlabel('Quantidade de Casos')
plt.ylabel('Frequência')
plt.yscale('log')  # Escala logarítmica devido à variação

plt.tight_layout()
plt.show()

print(f"🏆 Ranking dos estados por casos:")
for i, (estado, casos) in enumerate(casos_por_estado.head(5).items(), 1):
    print(f"   {i}º. {estado}: {casos:,} casos")

In [ ]:
# Análise de correlação entre variáveis climáticas e casos de dengue
variaveis_climaticas = [
    'precipitacao_media_mensal_uf',
    'temp_max_media_mensal_uf',
    'temp_min_media_mensal_uf',
    'umidade_max_media_mensal_uf',
    'umidade_min_media_mensal_uf'
]

# Matriz de correlação
plt.figure(figsize=(12, 8))
correlacao = df[variaveis_climaticas + ['Quantidade de Casos']].corr()
sns.heatmap(correlacao, annot=True, cmap='RdYlBu_r', center=0,
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Correlação entre Variáveis Climáticas e Casos de Dengue')
plt.tight_layout()
plt.show()

print("🌡️ Correlações com a quantidade de casos:")
correlacoes_casos = correlacao['Quantidade de Casos'].drop('Quantidade de Casos').sort_values(key=abs, ascending=False)
for var, corr in correlacoes_casos.items():
    print(f"   • {var}: {corr:.3f}")

## 📋 Conclusões da Análise Inicial

**Principais insights encontrados:**

✅ **Estrutura dos Dados:**
- Dataset estruturado em formato CSV com 3.536 registros
- Período: 2014-2025 (dados mensais por estado)
- Sem valores ausentes
- Variáveis climáticas e de saneamento disponíveis

✅ **Padrões Temporais:**
- Sazonalidade clara nos casos de dengue
- Picos geralmente nos primeiros meses do ano
- Variação significativa entre anos

✅ **Distribuição Geográfica:**
- Concentração de casos em alguns estados específicos
- Necessidade de trabalhar em nível municipal (próximo passo)

✅ **Relações com Clima:**
- Correlações identificadas entre variáveis climáticas e casos
- Temperatura e umidade parecem influenciar nos casos

**Próximos Passos:**
1. Processamento e enriquecimento dos dados
2. Criação de features de lag e médias móveis
3. Desenvolvimento dos modelos de ML